cdk_mate turns Python objects into `cdk` command lines. This page builds the command lines without running them, so every output below is exactly what would be passed to the CDK CLI. It needs no AWS account and no `cdk` CLI. How a command runs (credentials, directory, logging) is on [CDK CLI in Python](../01-CDK-CLI-In-Python/index.rst).

## One Class per Command

In [1]:
import cdk_mate.api as cdk_mate

In [2]:
cmd = cdk_mate.cli.Deploy(
    stacks=["MyStack"],
    require_approval="never",
)
cmd.to_args()

['cdk', 'deploy', 'MyStack', '--require-approval', 'never']

A field you do not set passes nothing, so the CLI uses its own default.

In [3]:
cdk_mate.cli.Deploy().to_args()

['cdk', 'deploy']

## How a Value Becomes an Argument

`True` passes `--name`, and `False` passes `--no-name`, which turns off an option that is on by default:

In [4]:
cdk_mate.cli.Deploy(rollback=True).to_args()

['cdk', 'deploy', '--rollback']

In [5]:
cdk_mate.cli.Deploy(rollback=False).to_args()

['cdk', 'deploy', '--no-rollback']

`0` is a real value and is passed; `None` and `""` pass nothing.

In [6]:
cdk_mate.cli.Deploy(concurrency=0).to_args()

['cdk', 'deploy', '--concurrency', '0']

In [7]:
cdk_mate.cli.Deploy(change_set_name="").to_args()

['cdk', 'deploy']

A list repeats the option, and a dict becomes `KEY=VALUE` pairs:

In [8]:
cdk_mate.cli.Deploy(notification_arns=["arn:a", "arn:b"]).to_args()

['cdk',
 'deploy',
 '--notification-arns',
 'arn:a',
 '--notification-arns',
 'arn:b']

In [9]:
cdk_mate.cli.Deploy(parameters={"MyStack:Env": "dev"}).to_args()

['cdk', 'deploy', '--parameters', 'MyStack:Env=dev']

A count option such as `verbose` repeats the flag:

In [10]:
cdk_mate.cli.Synth(verbose=2).to_args()

['cdk', 'synth', '--verbose', '--verbose']

Command options come first, so positional arguments follow the subcommand; global options such as `profile` follow them:

In [11]:
cdk_mate.cli.Diff(
    stacks=["MyStack"],
    fail=True,
    profile="my_profile",
).to_args()

['cdk', 'diff', 'MyStack', '--fail', '--profile', 'my_profile']

## Options Not Defined Yet, and the CLI Version

`extra_args` is appended verbatim, for an option this library does not define yet. `cdk_cmd` replaces the `cdk` executable, for example to pin a CLI version per project.

In [12]:
cdk_mate.cli.Deploy(
    stacks=["MyStack"],
    extra_args=["--tags", "team=data"],
).to_args()

['cdk', 'deploy', 'MyStack', '--tags', 'team=data']

In [13]:
cdk_mate.cli.Synth(
    cdk_cmd=["npx", "aws-cdk@2.1145.0"],
    quiet=True,
).to_args()

['npx', 'aws-cdk@2.1145.0', 'synth', '--quiet']

## A Stack Context

In [14]:
cdk_mate.to_camel("cdk-mate-stack1-dev")

'CdkMateStack1Dev'

In [15]:
cdk_mate.to_slug("cdk_mate_stack1_dev")

'cdk-mate-stack1-dev'

This page has no boto session, so it creates the contexts directly. `123456789012` is a placeholder account id.

In [16]:
stack1 = cdk_mate.StackCtx(
    construct_id="CdkMateStack1Dev",
    stack_name="cdk-mate-stack1-dev",
    aws_account_id="123456789012",
    aws_region="us-east-1",
)
stack2 = cdk_mate.StackCtx(
    construct_id="CdkMateStack2Dev",
    stack_name="cdk-mate-stack2-dev",
    aws_account_id="123456789012",
    aws_region="us-east-1",
)

In [17]:
stack1.to_stack_kwargs()

{'id': 'CdkMateStack1Dev',
 'stack_name': 'cdk-mate-stack1-dev',
 'env': Environment(account='123456789012', region='us-east-1')}

In [18]:
stack1.stack_console_url

'https://us-east-1.console.aws.amazon.com/cloudformation/home?region=us-east-1#/stacks?filteringStatus=active&filteringText=cdk-mate-stack1-dev&viewNested=true'

## Commands for Stacks: build_cmd

In [19]:
cdk_mate.build_cmd(cdk_mate.cli.Synth, [stack1]).to_args()

['cdk', 'synth', 'CdkMateStack1Dev']

In [20]:
cdk_mate.build_cmd(cdk_mate.cli.Diff, [stack1, stack2]).to_args()

['cdk', 'diff', 'CdkMateStack1Dev', 'CdkMateStack2Dev']

`prompt=False`, the default, makes a deploy or destroy run without waiting for a human: `--require-approval never` for deploy, `--force` for destroy.

In [21]:
cdk_mate.build_cmd(cdk_mate.cli.Deploy, [stack1]).to_args()

['cdk', 'deploy', 'CdkMateStack1Dev', '--require-approval', 'never']

In [22]:
cdk_mate.build_cmd(cdk_mate.cli.Destroy, [stack1]).to_args()

['cdk', 'destroy', 'CdkMateStack1Dev', '--force']

`prompt=True` passes neither, so the CLI asks:

In [23]:
cdk_mate.build_cmd(cdk_mate.cli.Deploy, [stack1], prompt=True).to_args()

['cdk', 'deploy', 'CdkMateStack1Dev']

Any other field goes through, and a field you set wins over the `prompt` default:

In [24]:
cdk_mate.build_cmd(
    cdk_mate.cli.Deploy,
    [stack1, stack2],
    require_approval="broadening",
    concurrency=2,
).to_args()

['cdk',
 'deploy',
 'CdkMateStack1Dev',
 'CdkMateStack2Dev',
 '--concurrency',
 '2',
 '--require-approval',
 'broadening']

An empty list is an error, because there is no stack to run:

In [25]:
try:
    cdk_mate.build_cmd(cdk_mate.cli.Deploy, [])
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: stack_ctx_list is empty, there is no stack to run


## One Stack or Many

## Check Before You Run

Since building and running are separate, you can print the command line first, and run the same object once it looks right:

In [26]:
cmd = cdk_mate.build_cmd(cdk_mate.cli.Deploy, [stack1, stack2])
print(" ".join(cmd.to_args()))

cdk deploy CdkMateStack1Dev CdkMateStack2Dev --require-approval never
